In [12]:
import os
import numpy as np
import czifile
import imageio

input_folder = "../data/input"
output_folder = "../data/output"

os.makedirs(output_folder, exist_ok=True)


def normalize_for_debug(img):
    """Optional: only for visualization, not ML"""
    img = img.astype(float)
    if img.max() > img.min():
        img = (img - img.min()) / (img.max() - img.min())
    return (img * 255).astype(np.uint8)


for file in os.listdir(input_folder):
    if not file.lower().endswith(".czi"):
        continue

    print(f"\nProcessing: {file}")

    path = os.path.join(input_folder, file)

    with czifile.CziFile(path) as czi:
        data = czi.asarray()

    # Remove singleton dimensions
    data = np.squeeze(data)

    print("Shape after squeeze:", data.shape)

    base = os.path.splitext(file)[0]

    # -------------------------------
    # CASE 1: 5D → (T, C, Z, Y, X)
    # -------------------------------
    if data.ndim == 5:
        T, C, Z, Y, X = data.shape

        for t in range(T):
            for c in range(C):
                # Max projection over Z (best default for ML)
                frame = data[t, c].max(axis=0)

                out = os.path.join(output_folder, f"{base}_t{t}_c{c}.tif")
                imageio.imwrite(out, frame.astype(data.dtype))

                print("Saved:", out)

    # -------------------------------
    # CASE 2: 4D → could be:
    # (T, C, Y, X) OR (C, Z, Y, X)
    # -------------------------------
    elif data.ndim == 4:
        d0, d1, d2, d3 = data.shape

        # Heuristic: if second dim large → likely Z
        if d1 > 10:
            print("Interpreting as (C, Z, Y, X)")
            C, Z, Y, X = data.shape

            for c in range(C):
                frame = data[c].max(axis=0)

                out = os.path.join(output_folder, f"{base}_c{c}.tif")
                imageio.imwrite(out, frame.astype(data.dtype))

                print("Saved:", out)

        else:
            print("Interpreting as (T, C, Y, X)")
            T, C, Y, X = data.shape

            for t in range(T):
                for c in range(C):
                    frame = data[t, c]

                    out = os.path.join(output_folder, f"{base}_t{t}_c{c}.tif")
                    imageio.imwrite(out, frame.astype(data.dtype))

                    print("Saved:", out)

    # -------------------------------
    # CASE 3: 3D → (C, Y, X) or (Z, Y, X)
    # -------------------------------
    elif data.ndim == 3:
        d0, d1, d2 = data.shape

        if d0 <= 4:
            print("Interpreting as (C, Y, X)")
            for c in range(d0):
                frame = data[c]

                out = os.path.join(output_folder, f"{base}_c{c}.tif")
                imageio.imwrite(out, frame.astype(data.dtype))

                print("Saved:", out)

        else:
            print("Interpreting as (Z, Y, X)")
            frame = data.max(axis=0)

            out = os.path.join(output_folder, f"{base}.tif")
            imageio.imwrite(out, frame.astype(data.dtype))

            print("Saved:", out)

    # -------------------------------
    # CASE 4: 2D → (Y, X)
    # -------------------------------
    elif data.ndim == 2:
        print("2D image")

        out = os.path.join(output_folder, f"{base}.tif")
        imageio.imwrite(out, data.astype(data.dtype))

        print("Saved:", out)

    else:
        print("Unsupported shape:", data.shape)


Processing: EGFP-RAB7a+mCh-Perox-01.czi
Shape after squeeze: (2, 66, 102, 512)
Interpreting as (C, Z, Y, X)
Saved: ../data/output/EGFP-RAB7a+mCh-Perox-01_c0.tif
Saved: ../data/output/EGFP-RAB7a+mCh-Perox-01_c1.tif
